# Weekly Analytics Brief — Data Pull

Builds the numbers for the weekly brief. Sections, in build order:
1. **Acquisitions & Churn** (from the daily sheet) — OLJ vs OT, net subscription change
2. GA4 (users, sessions, page views, top articles) — *not built yet*
3. CMS (top articles detail, if GA4 doesn't cover it) — *not built yet*

Run top to bottom. The last cell prints the numbers in the same shape as the weekly brief template,
so you can eyeball it before we wire it into the doc.


## Dependencies

Run this once per Colab session (installs are wiped when the runtime resets).

In [1]:
!pip install -q google-api-python-client google-auth google-auth-oauthlib google-analytics-data python-docx


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import pandas as pd
import datetime as dt

# --- Week window ---
# Auto-detects the most recently COMPLETED Monday-Sunday week based on today's date.
# To check a specific past week instead, uncomment the override line below and edit it.
today = dt.datetime.combine(dt.date.today(), dt.time.min)
WEEK_END = today - dt.timedelta(days=today.weekday() + 1)  # most recent Sunday before today
# WEEK_END = dt.datetime(2026, 9, 20)  # <- uncomment + edit to override with a specific week

WEEK_START = WEEK_END - dt.timedelta(days=6)  # the Monday of that week
PREV_WEEK_END = WEEK_START - dt.timedelta(days=1)
PREV_WEEK_START = PREV_WEEK_END - dt.timedelta(days=6)

assert WEEK_END.weekday() == 6, "WEEK_END must be a Sunday"

print(f"This week:  {WEEK_START:%Y-%m-%d} (Mon) → {WEEK_END:%Y-%m-%d} (Sun)")
print(f"Last week:  {PREV_WEEK_START:%Y-%m-%d} (Mon) → {PREV_WEEK_END:%Y-%m-%d} (Sun)")


This week:  2026-09-14 (Mon) → 2026-09-20 (Sun)
Last week:  2026-09-07 (Mon) → 2026-09-13 (Sun)


## 0. Read the source sheet (read-only)

Reads values straight from the live Google Sheet via the Sheets API — **no file is downloaded, and
nothing is ever written back**. Uses the `spreadsheets.readonly` OAuth scope, which has no write
methods at all, so there's no code path here that could edit the sheet even by mistake.

**One-time setup** (not part of the weekly run):
1. In Google Cloud Console, create a **service account** and download its JSON key.
2. Share the Google Sheet with that service account's email (looks like
   `something@project-id.iam.gserviceaccount.com`) as **Viewer**.
3. For GitHub Actions: store the JSON key's contents as a repo secret (e.g. `GSHEET_SA_KEY`); the
   workflow writes it to `service_account.json` at run time before this notebook runs. Locally, just
   save the key as `service_account.json` next to this notebook.


In [3]:
import os

SPREADSHEET_ID = "11WU-b3nmvyPlO0fX9VycKgObr-v5-hXTN6ieCv2TOoA"
SERVICE_ACCOUNT_FILE = "service_account.json"

def read_sheet_rows_api(spreadsheet_id, sheet_name, service_account_file, last_col="R", last_row=5000):
    """Reads a tab's rows (from column B, row 3, to last_col/last_row) via the Sheets API.
    Read-only scope — .readonly has no update/append/clear methods, so this can only ever read.
    Returns a list of rows; each row is a list of raw cell values (UNFORMATTED_VALUE, so dates come
    back as serial numbers, same convention Excel uses)."""
    from google.oauth2 import service_account
    from googleapiclient.discovery import build

    creds = service_account.Credentials.from_service_account_file(
        service_account_file,
        scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],  # read-only, no write methods exist on this scope
    )
    service = build("sheets", "v4", credentials=creds)
    result = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"'{sheet_name}'!B3:{last_col}{last_row}",
        valueRenderOption="UNFORMATTED_VALUE",
    ).execute()
    return result.get("values", [])

def read_sheet_rows_local(path, sheet_name):
    """Fallback for local testing when service_account.json isn't set up yet: reads the same
    B3:R-range shape out of the uploaded snapshot file, so downstream code is identical either way."""
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb[sheet_name]
    rows = []
    for r in range(3, ws.max_row + 1):
        row = [ws.cell(row=r, column=c).value for c in range(2, 19)]  # B..R
        if any(v is not None for v in row):
            rows.append(row)
    return rows

USE_API = os.path.exists(SERVICE_ACCOUNT_FILE)
if USE_API:
    print("Reading live sheet via Sheets API (read-only)...")
else:
    print("[no service_account.json found — reading local snapshot instead for now]")


[no service_account.json found — reading local snapshot instead for now]


## 1. Acquisitions & Churn

Rows come from section 0 above (live via API, or local snapshot as fallback) — column indices below
are relative to column B (B=0), matching the B:R range fetched there.

Column layout (confirmed against the sheet's own pre-built totals, so we trust these instead of
re-summing raw columns by hand):
- **Acquisitions**: O = grand total, P = OLJ Basic total, Q = OLJ Premium total, R = OT total → OLJ = P+Q
- **Churns**: L = grand total, M = OLJ Basic total, N = OLJ Premium total, O = OT total → OLJ = M+N


In [4]:
def excel_serial_to_datetime(v):
    """API returns dates as Excel-style serial numbers (UNFORMATTED_VALUE); openpyxl local fallback
    already gives real datetimes. Handle both."""
    if isinstance(v, dt.datetime):
        return v
    if isinstance(v, (int, float)):
        return dt.datetime(1899, 12, 30) + dt.timedelta(days=v)
    return None

def week_totals_from_rows(rows, col_idxs, start, end):
    """col_idxs are 0-based, relative to column B (date is index 0)."""
    totals = {c: 0 for c in col_idxs}
    for row in rows:
        if not row:
            continue
        d = excel_serial_to_datetime(row[0]) if len(row) > 0 else None
        if d and start <= d <= end:
            for c in col_idxs:
                v = row[c] if c < len(row) else 0
                totals[c] += v or 0
    return totals

# 0-based offsets from column B: O=13, P=14, Q=15, R=16 (Acquisitions); L=10, M=11, N=12, O=13 (Churns)
ACQ_COLS = [13, 14, 15, 16]
CHU_COLS = [10, 11, 12, 13]

def acq_churn_week(start, end):
    if USE_API:
        acq_rows = read_sheet_rows_api(SPREADSHEET_ID, "Acquisitions", SERVICE_ACCOUNT_FILE)
        chu_rows = read_sheet_rows_api(SPREADSHEET_ID, "Churns", SERVICE_ACCOUNT_FILE)
    else:
        acq_rows = read_sheet_rows_local("Daily_sheet_Acquisitions_Churns.xlsx", "Acquisitions")
        chu_rows = read_sheet_rows_local("Daily_sheet_Acquisitions_Churns.xlsx", "Churns")

    acq = week_totals_from_rows(acq_rows, ACQ_COLS, start, end)
    chu = week_totals_from_rows(chu_rows, CHU_COLS, start, end)

    olj_new = acq[14] + acq[15]
    ot_new  = acq[16]
    olj_churn = chu[11] + chu[12]
    ot_churn  = chu[13]

    return {
        "olj_new": olj_new, "olj_churn": olj_churn, "olj_net": olj_new - olj_churn,
        "ot_new": ot_new,   "ot_churn": ot_churn,   "ot_net": ot_new - ot_churn,
    }

this_week = acq_churn_week(WEEK_START, WEEK_END)
last_week = acq_churn_week(PREV_WEEK_START, PREV_WEEK_END)

this_week, last_week


({'olj_new': 64,
  'olj_churn': 104,
  'olj_net': -40,
  'ot_new': 25,
  'ot_churn': 13,
  'ot_net': 12},
 {'olj_new': 60,
  'olj_churn': 87,
  'olj_net': -27,
  'ot_new': 24,
  'ot_churn': 20,
  'ot_net': 4})

In [5]:
def net_row(this_w, last_w, prefix):
    return {
        "This week": f"{this_w[f'{prefix}_net']:+d} ({this_w[f'{prefix}_new']} new / {this_w[f'{prefix}_churn']} churn)",
        "Last week": f"{last_w[f'{prefix}_net']:+d} ({last_w[f'{prefix}_new']} new / {last_w[f'{prefix}_churn']} churn)",
        "WoW (Δ net)": f"{this_w[f'{prefix}_net'] - last_w[f'{prefix}_net']:+d}",
    }

olj_table = pd.DataFrame([net_row(this_week, last_week, "olj")], index=["Net Subscription Change (new − churn)"])
ot_table  = pd.DataFrame([net_row(this_week, last_week, "ot")],  index=["Net Subscription Change (new − churn)"])

print("OLJ")
display(olj_table)
print("\nOT")
display(ot_table)


OLJ


,This week,Last week,WoW (Δ net)
Net Subscription Change (new − churn),-40 (64 new / 104 churn),-27 (60 new / 87 churn),-13



OT


,This week,Last week,WoW (Δ net)
Net Subscription Change (new − churn),+12 (25 new / 13 churn),+4 (24 new / 20 churn),+8


## 2. New accounts (OLJ vs OT)

Simpler than the CMS scraping approach — this is already tracked daily in the **`Accounts created OLJ OT`**
tab of the same spreadsheet (columns: Date, Accounts created in period OT, Accounts created in period OLJ).
Reuses the exact same read-only connection from section 0 — no separate login needed.

Note: this whole report is **OLJ-specific**, matching the GA4 stream filter — OT is kept alongside for
reference since the sheet already tracks both, but OLJ is the number that matters for the brief.


In [6]:
def read_accounts_created_rows():
    if USE_API:
        return read_sheet_rows_api_custom(SPREADSHEET_ID, "Accounts created OLJ OT",
                                            SERVICE_ACCOUNT_FILE, first_col="A", last_col="D")
    else:
        import openpyxl
        wb = openpyxl.load_workbook("Daily_sheet_Acquisitions_Churns.xlsx", data_only=True)
        ws = wb["Accounts created OLJ OT"]
        rows = []
        for r in range(2, ws.max_row + 1):
            row = [ws.cell(row=r, column=c).value for c in range(1, 5)]  # A..D
            if any(v is not None for v in row):
                rows.append(row)
        return rows

def read_sheet_rows_api_custom(spreadsheet_id, sheet_name, service_account_file, first_col="A", last_col="D", last_row=5000):
    """Same idea as read_sheet_rows_api in section 0, but this tab\'s date column is A, not B,
    so it needs its own starting column."""
    from google.oauth2 import service_account as sa
    from googleapiclient.discovery import build

    creds = sa.Credentials.from_service_account_file(
        service_account_file, scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"]
    )
    service = build("sheets", "v4", credentials=creds)
    result = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"\'{sheet_name}\'!{first_col}2:{last_col}{last_row}",
        valueRenderOption="UNFORMATTED_VALUE",
    ).execute()
    return result.get("values", [])

def accounts_created_week(start, end):
    rows = read_accounts_created_rows()
    ot_total, olj_total = 0, 0
    for row in rows:
        if not row:
            continue
        d = excel_serial_to_datetime(row[0]) if len(row) > 0 else None
        if d and start <= d <= end:
            ot_total += (row[1] if len(row) > 1 and row[1] else 0)
            olj_total += (row[2] if len(row) > 2 and row[2] else 0)
    return {"olj_new_accounts": int(olj_total), "ot_new_accounts": int(ot_total)}

this_week_accounts = accounts_created_week(WEEK_START, WEEK_END)
last_week_accounts = accounts_created_week(PREV_WEEK_START, PREV_WEEK_END)

this_week_accounts, last_week_accounts


({'olj_new_accounts': 0, 'ot_new_accounts': 0},
 {'olj_new_accounts': 68, 'ot_new_accounts': 14})

## 3. GA4 — Users, Sessions, Page views, Top articles, Countries, Sources

Uses the **GA4 Data API** (`analytics.readonly` scope — same read-only pattern as sections 0 and 2)
with the **same service account** from section 0 — you just need to also grant it access on the GA4
property itself (one extra step below), no new key needed.

**One-time setup:**
1. In the same Google Cloud project as before, enable the **"Google Analytics Data API"**
   (Console → search "Google Analytics Data API" → Enable).
2. In GA4: **Admin → Property Access Management** (for the property, not the account) → **+** →
   add the service account's email (the same `...iam.gserviceaccount.com` address from section 0) →
   role **Viewer**.
3. Find your **Property ID**: GA4 Admin → Property Details → it's a plain number like `123456789`
   (not the "G-XXXXXXX" measurement ID — that's a different thing). Fill it into `GA4_PROPERTY_ID` below.

⚠️ Same caveat as CMS: **untested against a real property** — I don't have GA4 access from here. If
`GA4_PROPERTY_ID` is left as the placeholder, or the service account isn't authorized yet, this section
runs on clearly-labeled **example data** instead, so you can see the shape of the output now and swap
in real numbers once it's connected.

**Fallback while waiting on Property Access Management approval:** logs in as *you* instead of the
service account. First run opens a browser for a one-time "Allow" click; after that it's silent
(the refresh token in `token.json` handles every run after, including in GitHub Actions later — no
repeated clicking).

**One-time setup for this fallback specifically:**
1. Same Google Cloud project → **APIs & Services → Credentials → Create Credentials → OAuth client ID**
   → Application type **Desktop app** → Create.
2. Download its JSON → save as `oauth_client_secret.json` next to this notebook (different file from
   `service_account.json` — don't mix them up).
3. Run the next cell. A browser tab opens asking you to log in and approve — do that once.

⚠️ One thing to know: if the OAuth app stays in **"Testing"** publishing status (Cloud Console →
OAuth consent screen), Google expires the refresh token after 7 days, which would silently break the
scheduled run a week in. Worth switching it to **"In production"** (or **"Internal"** if this is a
Google Workspace org) once you confirm the login flow works, so it keeps running unattended.


In [7]:
GA4_PROPERTY_ID = "328439412"  # GA4 Admin > Property Details
OAUTH_CLIENT_SECRET_FILE = "oauth_client_secret.json"  # from Cloud Console > Credentials > OAuth client ID (Desktop app)
OAUTH_TOKEN_FILE = "token.json"  # created automatically after your first approval; reused silently after that
GA4_SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

def get_ga4_credentials():
    """Prefers the service account (once Property Access Management is granted); falls back to
    logging in as you via OAuth, which only needs a browser click on the very first run."""
    if os.path.exists(SERVICE_ACCOUNT_FILE):
        from google.oauth2 import service_account as ga_service_account
        return ga_service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=GA4_SCOPES)

    if os.path.exists(OAUTH_CLIENT_SECRET_FILE):
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google_auth_oauthlib.flow import InstalledAppFlow

        creds = None
        if os.path.exists(OAUTH_TOKEN_FILE):
            creds = Credentials.from_authorized_user_file(OAUTH_TOKEN_FILE, GA4_SCOPES)
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(OAUTH_CLIENT_SECRET_FILE, GA4_SCOPES)
                creds = flow.run_local_server(port=0)  # opens your browser -- click Allow once
            with open(OAUTH_TOKEN_FILE, "w") as f:
                f.write(creds.to_json())
        return creds

    return None

GA4_READY = (
    (os.path.exists(SERVICE_ACCOUNT_FILE) or os.path.exists(OAUTH_CLIENT_SECRET_FILE))
    and GA4_PROPERTY_ID != "REPLACE_WITH_YOUR_PROPERTY_ID"
)

if GA4_READY:
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.analytics.data_v1beta.types import (
        RunReportRequest, DateRange, Metric, Dimension, OrderBy,
        FilterExpression, Filter,
    )
    ga4_creds = get_ga4_credentials()
    ga4_client = BetaAnalyticsDataClient(credentials=ga4_creds)

    # Every table in the report is filtered to OLJ streams only (name contains "olj",
        # case-insensitive, so it matches "OLJ Website" / "olj android" / "OLJ iOS" etc.)
    olj_stream_filter = FilterExpression(
        filter=Filter(
            field_name="streamName",
            string_filter=Filter.StringFilter(
                match_type=Filter.StringFilter.MatchType.CONTAINS,
                value="olj",
                case_sensitive=False,
            )
        )
    )

    def ga4_report(start, end, metrics, dimensions=None, order_by_metric=None, limit=None,
                   extra_filter=None):
        dim_filter = olj_stream_filter
        if extra_filter is not None:
            from google.analytics.data_v1beta.types import FilterExpressionList
            dim_filter = FilterExpression(
                and_group=FilterExpressionList(expressions=[olj_stream_filter, extra_filter])
            )
        request = RunReportRequest(
            property=f"properties/{GA4_PROPERTY_ID}",
            date_ranges=[DateRange(start_date=start.strftime("%Y-%m-%d"), end_date=end.strftime("%Y-%m-%d"))],
            metrics=[Metric(name=m) for m in metrics],
            dimensions=[Dimension(name=d) for d in (dimensions or [])],
            dimension_filter=dim_filter,
            limit=limit,
        )
        if order_by_metric:
            request.order_bys = [OrderBy(metric=OrderBy.MetricOrderBy(metric_name=order_by_metric), desc=True)]
        resp = ga4_client.run_report(request)
        cols = [d.name for d in resp.dimension_headers] + [m.name for m in resp.metric_headers]
        rows = []
        for row in resp.rows:
            rows.append([v.value for v in row.dimension_values] + [v.value for v in row.metric_values])
        return pd.DataFrame(rows, columns=cols)

    print(f"GA4 ready -- property {GA4_PROPERTY_ID}")
else:
    print("[GA4 not connected yet -- using example data for this section. "
          "Set GA4_PROPERTY_ID and make sure service_account.json is present + authorized on the property.]")


[GA4 not connected yet -- using example data for this section. Set GA4_PROPERTY_ID and make sure service_account.json is present + authorized on the property.]


### 3a. Scorecard metrics — Users, Sessions, Page views

In [8]:
def ga4_scorecard(start, end):
    if GA4_READY:
        df = ga4_report(start, end, metrics=["totalUsers", "sessions", "screenPageViews"])
        row = df.iloc[0]
        return {"users": int(row["totalUsers"]), "sessions": int(row["sessions"]), "page_views": int(row["screenPageViews"])}
    else:
        # Example data, roughly matching the shape from the template we built earlier
        import random
        random.seed(hash(start))
        base = 42000 if start == WEEK_START else 39900
        return {"users": base, "sessions": int(base * 1.39), "page_views": int(base * 2.66)}

this_week_ga4 = ga4_scorecard(WEEK_START, WEEK_END)
last_week_ga4 = ga4_scorecard(PREV_WEEK_START, PREV_WEEK_END)

def wow_pct(this_v, last_v):
    if last_v == 0:
        return "n/a"
    pct = (this_v - last_v) / last_v * 100
    arrow = "\u25b2" if pct >= 0 else "\u25bc"
    return f"{arrow} {abs(pct):.0f}%"

def signed_wow_pct(this_v, last_v):
    """For metrics that can be negative (like Net Subscription Change): direction is
    based on whether the value actually improved (this_v >= last_v), not the raw sign of
    the % change -- plain division misleads here, since -27 -> -40 is worse but naive
    math (dividing two negatives) would show it as a positive-looking \"+48%\"."""
    if last_v == 0:
        return "n/a"
    pct = abs(this_v - last_v) / abs(last_v) * 100
    arrow = "\u25b2" if this_v >= last_v else "\u25bc"
    return f"{arrow} {pct:.0f}%"

scorecard_df = pd.DataFrame([
    {"Metric": "Users", "This week": this_week_ga4["users"], "Last week": last_week_ga4["users"],
     "WoW": wow_pct(this_week_ga4["users"], last_week_ga4["users"])},
    {"Metric": "Sessions", "This week": this_week_ga4["sessions"], "Last week": last_week_ga4["sessions"],
     "WoW": wow_pct(this_week_ga4["sessions"], last_week_ga4["sessions"])},
    {"Metric": "Page views", "This week": this_week_ga4["page_views"], "Last week": last_week_ga4["page_views"],
     "WoW": wow_pct(this_week_ga4["page_views"], last_week_ga4["page_views"])},
]).set_index("Metric")

scorecard_df


,This week,Last week,WoW
Metric,,,
Users,42000,39900,▲ 5%
Sessions,58379,55460,▲ 5%
Page views,111720,106134,▲ 5%


### 3b. Top 10 articles

In [9]:
def ga4_top_articles(start, end, n=10):
    if GA4_READY:
        df = ga4_report(start, end, metrics=["screenPageViews"], dimensions=["pageTitle"],
                         order_by_metric="screenPageViews", limit=n)
        df["screenPageViews"] = df["screenPageViews"].astype(int)
        return df.rename(columns={"pageTitle": "Article", "screenPageViews": "Views"})
    else:
        return pd.DataFrame({
            "Article": [f"Example article title {i}" for i in range(1, n + 1)],
            "Views": sorted([4200, 3100, 2650, 2200, 1800, 1600, 1400, 1250, 1100, 950], reverse=True)[:n],
        })

top_articles = ga4_top_articles(WEEK_START, WEEK_END)
top_articles


,Article,Views
0,Example article title 1,4200
1,Example article title 2,3100
2,Example article title 3,2650
3,Example article title 4,2200
4,Example article title 5,1800
5,Example article title 6,1600
6,Example article title 7,1400
7,Example article title 8,1250
8,Example article title 9,1100
9,Example article title 10,950


### 3e. Top 10 articles (web + app views summed by Article ID)

Web events write the article ID to the `articleid` custom dimension; app events write it to a
**separate** `article_id` dimension — so this pulls both independently (each already filtered to OLJ
streams) and merges them by matching ID value, summing views across web + app for the same article
before ranking.


In [10]:
import re

ARTICLE_ID_PATTERN = re.compile(r"^\d{6,7}$")  # valid article IDs only -- drops "(not set)" and junk

def ga4_top_articles_by_id(start, end, n=10):
    if GA4_READY:
        # pageTitle pulled in the SAME query as the ID -- not a separate lookup call
        web = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:articleid", "pageTitle"])
        app = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:article_id", "pageTitle"])

        web = web.rename(columns={"customEvent:articleid": "article_id", "screenPageViews": "views"})
        app = app.rename(columns={"customEvent:article_id": "article_id", "screenPageViews": "views"})

        combined = pd.concat([web, app], ignore_index=True)
        combined["views"] = combined["views"].astype(int)

        # Keep only valid numeric article IDs (6 or 7 digits) -- excludes "(not set)", blanks, junk
        valid = combined["article_id"].astype(str).str.strip().apply(lambda v: bool(ARTICLE_ID_PATTERN.match(v)))
        combined = combined[valid]

        summed = combined.groupby("article_id", as_index=False)["views"].sum()
        top = summed.sort_values("views", ascending=False).head(n).reset_index(drop=True)

        # Rank by ID first (above), THEN attach a title per ID -- the most frequent title seen for
        # that ID across both web and app rows (guards against a stray differently-formatted title)
        titles = (
            combined[combined["article_id"].isin(top["article_id"])]
            .groupby("article_id")["pageTitle"]
            .agg(lambda s: s.value_counts().idxmax() if len(s) else "")
            .to_dict()
        )
        top["Article"] = top["article_id"].map(titles)
        return top.rename(columns={"article_id": "Article ID", "views": "Views"})[["Article", "Article ID", "Views"]]
    else:
        return pd.DataFrame({
            "Article": [f"Example article title {i}" for i in range(1, n + 1)],
            "Article ID": [f"15481{i:02d}" for i in range(1, n + 1)],
            "Views": sorted([4200, 3100, 2650, 2200, 1800, 1600, 1400, 1250, 1100, 950], reverse=True)[:n],
        })

top_articles_by_id = ga4_top_articles_by_id(WEEK_START, WEEK_END)
top_articles_by_id


,Article,Article ID,Views
0,Example article title 1,1548101,4200
1,Example article title 2,1548102,3100
2,Example article title 3,1548103,2650
3,Example article title 4,1548104,2200
4,Example article title 5,1548105,1800
5,Example article title 6,1548106,1600
6,Example article title 7,1548107,1400
7,Example article title 8,1548108,1250
8,Example article title 9,1548109,1100
9,Example article title 10,1548110,950


### 3c. Top countries & top sources

In [11]:
def ga4_top_dimension(start, end, dimension, n=5):
    if GA4_READY:
        df = ga4_report(start, end, metrics=["sessions"], dimensions=[dimension],
                         order_by_metric="sessions", limit=n)
        df["sessions"] = df["sessions"].astype(int)
        total = df["sessions"].sum()
        df["share"] = (df["sessions"] / total * 100).round(0).astype(int).astype(str) + "%"
        return df.rename(columns={dimension: dimension.capitalize(), "sessions": "Sessions", "share": "Share"})
    else:
        example = {
            "country": pd.DataFrame({"Country": ["Lebanon", "France", "USA", "Canada", "UAE"],
                                       "Sessions": [24800, 8900, 6200, 3100, 2400],
                                       "Share": ["42%", "15%", "11%", "5%", "4%"]}),
            "sessionSource": pd.DataFrame({"Sessionsource": ["(direct)", "google", "olj", "website", "bing", "CMS-9", "(not set)"],
                                       "Sessions": [15200, 11800, 9400, 5900, 3100, 1900, 1200],
                                       "Share": ["31%", "24%", "19%", "12%", "6%", "4%", "2%"]}),
        }
        return example[dimension]

top_countries = ga4_top_dimension(WEEK_START, WEEK_END, "country")
top_sources = ga4_top_dimension(WEEK_START, WEEK_END, "sessionSource")

print("Top countries")
display(top_countries)
print("\nTop sources")
display(top_sources)


Top countries


,Country,Sessions,Share
0,Lebanon,24800,42%
1,France,8900,15%
2,USA,6200,11%
3,Canada,3100,5%
4,UAE,2400,4%



Top sources


,Sessionsource,Sessions,Share
0,(direct),15200,31%
1,google,11800,24%
2,olj,9400,19%
3,website,5900,12%
4,bing,3100,6%
5,CMS-9,1900,4%
6,(not set),1200,2%


### 3f. Sources mapped into categories (OLJ)

In [12]:
MAIN_CATS = ["Direct", "Search Engines", "Social Networks", "AI Assistants", "Internal/Newsletters", "Other"]

MAIN_COLORS = {
    "Direct":               "#4285F4",
    "Search Engines":       "#00BFA5",
    "Social Networks":      "#9C27B0",
    "AI Assistants":        "#FFA726",
    "Internal/Newsletters": "#FF6B9D",
    "Other":                "#9E9E9E",
}

DIRECT = {"(direct)"}

SEARCH_ENGINES = {
    "google", "news.google.com", "bing", "ecosia.org", "qwant.com", "duckduckgo",
    "fr.search.yahoo.com", "yahoo", "search.brave.com", "yandex", "ya.ru", "startpage.com",
}

SOCIAL_NETWORKS = {
    "m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
    "mobile.facebook.com", "facebook", "l.instagram.com", "ig", "instagram",
    "instagram.com", "later-linkinbio", "linkin.bio", "t.co", "linkedin.com",
    "lnkd.in", "go.bsky.app", "l.threads.com", "twitter", "x.com", "threads",
    "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
    "web.whatsapp.com", "cms-46", "reddit.com", "fb", "flipboard", "flipboard.com",
}

AI_ASSISTANTS = {
    "chatgpt.com", "perplexity.ai", "gemini.google.com", "perplexity", "copilot.com",
    "copilot.microsoft.com", "openai", "duck.ai", "chat.mistral.ai",
    "notebooklm.google.com", "claude.ai", "poe.com", "grok.com", "chat.qwen.ai",
    "doubao.com", "chat.z.ai", "felo.ai", "mammouth.ai", "you.com",
}

INTERNAL_NEWSLETTERS = {
    "mailchimp", "newsletter", "email", "website", "olj", "google-play", "olj.me",
    "olj.wael", "autopromoolj", "morningbrief", "hs_email", "brevo", "mailchi.mp",
    "us1.campaign-archive.com", "actito.be", "activetrail", "acumbamail", "omnisend",
    "wordfly", "newsletter_1", "newsletter_6b", "newsletter_paiementechouepp",
    "newsletter_preventif", "partenairesjamhour", "marketo",
    "gmi mailchimp integration prod list", "master list", "bundle", "nb",
}

def categorize_source(src):
    s = str(src).strip().lower()
    if s in DIRECT:
        return "Direct"
    if s in SEARCH_ENGINES:
        return "Search Engines"
    if s in SOCIAL_NETWORKS:
        return "Social Networks"
    if s in AI_ASSISTANTS:
        return "AI Assistants"
    if s in INTERNAL_NEWSLETTERS:
        return "Internal/Newsletters"
    return "Other"

def ga4_sources_by_category(start, end):
    if GA4_READY:
        # No limit -- need every distinct source to categorize correctly, not just a top-N slice
        df = ga4_report(start, end, metrics=["sessions"], dimensions=["sessionSource"], limit=100000)
        df["sessions"] = df["sessions"].astype(int)
        df["Category"] = df["sessionSource"].apply(categorize_source)
        totals = df.groupby("Category")["sessions"].sum().reindex(MAIN_CATS, fill_value=0)
    else:
        totals = pd.Series(
            {"Direct": 15200, "Search Engines": 13700, "Social Networks": 9400,
             "AI Assistants": 1200, "Internal/Newsletters": 9400, "Other": 3100}
        ).reindex(MAIN_CATS, fill_value=0)

    total = totals.sum()
    result = pd.DataFrame({"Category": MAIN_CATS, "Sessions": totals.values.astype(int)})
    result["Share"] = (result["Sessions"] / total * 100).round(0).astype(int).astype(str) + "%"
    result["Color"] = result["Category"].map(MAIN_COLORS)
    return result

sources_by_category = ga4_sources_by_category(WEEK_START, WEEK_END)
sources_by_category


,Category,Sessions,Share,Color
0,Direct,15200,29%,#4285F4
1,Search Engines,13700,26%,#00BFA5
2,Social Networks,9400,18%,#9C27B0
3,AI Assistants,1200,2%,#FFA726
4,Internal/Newsletters,9400,18%,#FF6B9D
5,Other,3100,6%,#9E9E9E


### 3d. App downloads — iOS vs Android

In [13]:
def ga4_app_downloads(start, end):
    if GA4_READY:
        from google.analytics.data_v1beta.types import Filter as _Filter, FilterExpression as _FE
        event_filter = _FE(filter=_Filter(field_name="eventName",
                                           string_filter=_Filter.StringFilter(value="app_download")))
        df = ga4_report(start, end, metrics=["eventCount"], dimensions=["platform"],
                         extra_filter=event_filter)
        counts = {row["platform"]: int(row["eventCount"]) for _, row in df.iterrows()}
        return {"ios": counts.get("iOS", 0), "android": counts.get("Android", 0)}
    else:
        return {"ios": 520, "android": 370} if start == WEEK_START else {"ios": 480, "android": 337}

this_week_downloads = ga4_app_downloads(WEEK_START, WEEK_END)
last_week_downloads = ga4_app_downloads(PREV_WEEK_START, PREV_WEEK_END)

downloads_df = pd.DataFrame([
    {"Metric": "App downloads (iOS / Android)",
     "This week": f"{sum(this_week_downloads.values())} ({this_week_downloads['ios']} / {this_week_downloads['android']})",
     "Last week": f"{sum(last_week_downloads.values())} ({last_week_downloads['ios']} / {last_week_downloads['android']})",
     "WoW": wow_pct(sum(this_week_downloads.values()), sum(last_week_downloads.values()))}
]).set_index("Metric")

downloads_df


,This week,Last week,WoW
Metric,,,
App downloads (iOS / Android),890 (520 / 370),817 (480 / 337),▲ 9%


## 4. Final report — OLJ Weekly Brief

Assembles everything above into the exact shape of the Weekly Analytics Brief template (scorecard,
top 5 articles, top countries/sources, one auto-flagged thing to watch) — OLJ only, since every GA4
query above is already filtered to OLJ streams and the sheet-based numbers use the OLJ columns.

The headline and "one thing to watch" line are auto-picked from whichever metric moved the most WoW
— not a substitute for someone actually reading the numbers, but a reasonable starting point each week.


In [14]:
from IPython.display import Markdown, display

def pct_change(this_v, last_v):
    if last_v == 0:
        return None
    return (this_v - last_v) / last_v * 100

dl_this_total = sum(this_week_downloads.values())
dl_last_total = sum(last_week_downloads.values())

movers = {
    "Users": pct_change(this_week_ga4["users"], last_week_ga4["users"]),
    "Sessions": pct_change(this_week_ga4["sessions"], last_week_ga4["sessions"]),
    "Page views": pct_change(this_week_ga4["page_views"], last_week_ga4["page_views"]),
    "App downloads": pct_change(dl_this_total, dl_last_total),
}

biggest_mover = max(movers, key=lambda k: abs(movers[k]) if movers[k] is not None else 0)
mv = movers[biggest_mover]
direction = "up" if mv is not None and mv >= 0 else "down"
headline = f"{biggest_mover} {direction} {abs(mv):.0f}% week-over-week." if mv is not None else "No clear standout metric this week."

net_delta = this_week["olj_net"] - last_week["olj_net"]
watch_lines = []
for name, pct in movers.items():
    if pct is not None and pct <= -10:
        watch_lines.append(f"{name} down {abs(pct):.0f}% week-over-week.")
if net_delta < 0:
    watch_lines.append(f"Net subscriptions worsened by {net_delta:+d} vs. last week ({this_week['olj_net']:+d} now).")
watch_line = " ".join(watch_lines) if watch_lines else "Nothing off-trend this week."

scorecard_md = f"""| Metric | This week | Last week | WoW |
|---|---|---|---|
| Users | {this_week_ga4['users']:,} | {last_week_ga4['users']:,} | {wow_pct(this_week_ga4['users'], last_week_ga4['users'])} |
| Sessions | {this_week_ga4['sessions']:,} | {last_week_ga4['sessions']:,} | {wow_pct(this_week_ga4['sessions'], last_week_ga4['sessions'])} |
| Page views | {this_week_ga4['page_views']:,} | {last_week_ga4['page_views']:,} | {wow_pct(this_week_ga4['page_views'], last_week_ga4['page_views'])} |
| New accounts | {this_week_accounts['olj_new_accounts']:,} | {last_week_accounts['olj_new_accounts']:,} | {wow_pct(this_week_accounts['olj_new_accounts'], last_week_accounts['olj_new_accounts'])} |
| Net Subscription Change (new \u2212 churn) | {this_week['olj_net']:+d} ({this_week['olj_new']} new / {this_week['olj_churn']} churn) | {last_week['olj_net']:+d} ({last_week['olj_new']} new / {last_week['olj_churn']} churn) | {signed_wow_pct(this_week['olj_net'], last_week['olj_net'])} |
| App downloads (iOS / Android) | {dl_this_total} ({this_week_downloads['ios']} / {this_week_downloads['android']}) | {dl_last_total} ({last_week_downloads['ios']} / {last_week_downloads['android']}) | {wow_pct(dl_this_total, dl_last_total)} |"""

articles_lines = "\n".join(
    f"{i+1}. {row['Article']} \u2014 {row['Views']:,} views"
    for i, row in top_articles_by_id.reset_index(drop=True).iterrows()
)

from itertools import zip_longest
country_recs = top_countries.to_dict("records")
category_recs = sources_by_category.to_dict("records")
traffic_rows_list = []
for c, s in zip_longest(country_recs, category_recs):
    left = f"{c['Country']} \u2014 {c['Share']}" if c else ""
    right = f"{s['Category']} \u2014 {s['Share']}" if s else ""
    traffic_rows_list.append(f"| {left} | {right} |")
traffic_rows = "\n".join(traffic_rows_list)
traffic_md = f"| Top countries | Sources by category |\n|---|---|\n{traffic_rows}"

report_md = f"""# Weekly Analytics Brief \u2014 OLJ
### Week of {WEEK_START:%b %d}\u2013{WEEK_END:%b %d, %Y}

**Headline:** {headline}

{scorecard_md}

## Top 10 articles this week
{articles_lines}

## Where traffic came from
{traffic_md}

## One thing to watch
{watch_line}
"""

display(Markdown(report_md))


# Weekly Analytics Brief — OLJ
### Week of Sep 14–Sep 20, 2026

**Headline:** App downloads up 9% week-over-week.

| Metric | This week | Last week | WoW |
|---|---|---|---|
| Users | 42,000 | 39,900 | ▲ 5% |
| Sessions | 58,379 | 55,460 | ▲ 5% |
| Page views | 111,720 | 106,134 | ▲ 5% |
| New accounts | 0 | 68 | ▼ 100% |
| Net Subscription Change (new − churn) | -40 (64 new / 104 churn) | -27 (60 new / 87 churn) | ▼ 48% |
| App downloads (iOS / Android) | 890 (520 / 370) | 817 (480 / 337) | ▲ 9% |

## Top 10 articles this week
1. Example article title 1 — 4,200 views
2. Example article title 2 — 3,100 views
3. Example article title 3 — 2,650 views
4. Example article title 4 — 2,200 views
5. Example article title 5 — 1,800 views
6. Example article title 6 — 1,600 views
7. Example article title 7 — 1,400 views
8. Example article title 8 — 1,250 views
9. Example article title 9 — 1,100 views
10. Example article title 10 — 950 views

## Where traffic came from
| Top countries | Sources by category |
|---|---|
| Lebanon — 42% | Direct — 29% |
| France — 15% | Search Engines — 26% |
| USA — 11% | Social Networks — 18% |
| Canada — 5% | AI Assistants — 2% |
| UAE — 4% | Internal/Newsletters — 18% |
|  | Other — 6% |

## One thing to watch
Net subscriptions worsened by -13 vs. last week (-40 now).


## 5. Export to Word + email it

Builds a `.docx` from the exact numbers already computed above (not a re-parse of the markdown —
built directly from the same variables, so it can't drift from what's shown in section 4), then
emails it as an attachment.

**Sending email — one-time setup (Gmail SMTP with an App Password, not your regular password):**
1. The sending account needs **2-Step Verification** turned on (myaccount.google.com/security).
2. Go to **myaccount.google.com/apppasswords** → create one for "Mail" → copy the 16-character password.
3. Set it via environment variables (`GMAIL_ADDRESS`, `GMAIL_APP_PASSWORD`) — same pattern as the
   other credentials in this notebook — or it'll prompt you interactively.

⚠️ **Untested against a real send from here** — same caveat as everything credential-related: I have
no way to verify this actually delivers until you run it with real values. If Gmail rejects the login,
double-check it's an **App Password**, not the account's normal password (regular passwords are
rejected outright once 2-Step Verification is on).


In [15]:
import os
import getpass
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.base import MIMEBase
from email.mime.text import MIMEText
from email import encoders

from itertools import zip_longest
from docx import Document

RECIPIENT_EMAIL = "dianafarhat@lorientlejour.com"

def build_brief_docx(path):
    doc = Document()
    doc.add_heading("Weekly Analytics Brief \u2014 OLJ", level=0)
    doc.add_paragraph(f"Week of {WEEK_START:%b %d}\u2013{WEEK_END:%b %d, %Y}")

    p = doc.add_paragraph()
    p.add_run("Headline: ").bold = True
    p.add_run(headline)

    doc.add_heading("Scorecard", level=2)
    table = doc.add_table(rows=1, cols=4)
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    hdr[0].text, hdr[1].text, hdr[2].text, hdr[3].text = "Metric", "This week", "Last week", "WoW"

    def add_row(metric, this_v, last_v, wow):
        row = table.add_row().cells
        row[0].text, row[1].text, row[2].text, row[3].text = metric, this_v, last_v, wow

    add_row("Users", f"{this_week_ga4['users']:,}", f"{last_week_ga4['users']:,}",
            wow_pct(this_week_ga4['users'], last_week_ga4['users']))
    add_row("Sessions", f"{this_week_ga4['sessions']:,}", f"{last_week_ga4['sessions']:,}",
            wow_pct(this_week_ga4['sessions'], last_week_ga4['sessions']))
    add_row("Page views", f"{this_week_ga4['page_views']:,}", f"{last_week_ga4['page_views']:,}",
            wow_pct(this_week_ga4['page_views'], last_week_ga4['page_views']))
    add_row("New accounts", f"{this_week_accounts['olj_new_accounts']:,}", f"{last_week_accounts['olj_new_accounts']:,}",
            wow_pct(this_week_accounts['olj_new_accounts'], last_week_accounts['olj_new_accounts']))
    add_row("Net Subscription Change (new \u2212 churn)",
            f"{this_week['olj_net']:+d} ({this_week['olj_new']} new / {this_week['olj_churn']} churn)",
            f"{last_week['olj_net']:+d} ({last_week['olj_new']} new / {last_week['olj_churn']} churn)",
            signed_wow_pct(this_week['olj_net'], last_week['olj_net']))
    add_row("App downloads (iOS / Android)",
            f"{dl_this_total} ({this_week_downloads['ios']} / {this_week_downloads['android']})",
            f"{dl_last_total} ({last_week_downloads['ios']} / {last_week_downloads['android']})",
            wow_pct(dl_this_total, dl_last_total))

    doc.add_heading("Top 10 articles this week", level=2)
    for i, row in top_articles_by_id.reset_index(drop=True).iterrows():
        doc.add_paragraph(f"{i+1}. {row['Article']} \u2014 {row['Views']:,} views")

    doc.add_heading("Where traffic came from", level=2)
    traffic_table = doc.add_table(rows=1, cols=2)
    traffic_table.style = "Light Grid Accent 1"
    thdr = traffic_table.rows[0].cells
    thdr[0].text, thdr[1].text = "Top countries", "Sources by category"
    for c, s in zip_longest(top_countries.to_dict("records"), sources_by_category.to_dict("records")):
        r = traffic_table.add_row().cells
        r[0].text = f"{c['Country']} \u2014 {c['Share']}" if c else ""
        r[1].text = f"{s['Category']} \u2014 {s['Share']}" if s else ""

    doc.add_heading("One thing to watch", level=2)
    doc.add_paragraph(watch_line)

    doc.save(path)
    return path

def send_email_with_attachment(recipient, subject, body, attachment_path):
    sender = os.environ.get("GMAIL_ADDRESS") or input("Sending Gmail address: ")
    app_password = os.environ.get("GMAIL_APP_PASSWORD") or getpass.getpass("Gmail App Password (not your normal password): ")

    msg = MIMEMultipart()
    msg["From"] = sender
    msg["To"] = recipient
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    with open(attachment_path, "rb") as f:
        part = MIMEBase("application", "octet-stream")
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header("Content-Disposition", f'attachment; filename="{os.path.basename(attachment_path)}"')
    msg.attach(part)

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(sender, app_password)
        server.send_message(msg)

docx_filename = f"Weekly_Analytics_Brief_OLJ_{WEEK_START:%Y%m%d}_{WEEK_END:%Y%m%d}.docx"  # relative -- saves next to the notebook, in Colab or anywhere else

try:
    build_brief_docx(docx_filename)
    print(f"Saved {docx_filename}")

    send_email_with_attachment(
        RECIPIENT_EMAIL,
        f"Weekly Analytics Brief \u2014 OLJ \u2014 Week of {WEEK_START:%b %d}",
        f"Attached: this week\'s OLJ analytics brief.\n\n{headline}",
        docx_filename,
    )
    print(f"Emailed to {RECIPIENT_EMAIL}")
except Exception as e:
    print(f"Export/email step failed -- paste this error back and I\'ll adjust.\n{type(e).__name__}: {e}")


Saved Weekly_Analytics_Brief_OLJ_20260914_20260920.docx
Export/email step failed -- paste this error back and I'll adjust.
StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.
